# Khung kiểm chứng A.2 và A.3

Đây là khung thí nghiệm CHƯA có kết quả, không phải bài nộp hoàn chỉnh. Nhóm tự đưa phần cài đặt của mình vào cell trống dưới đây. Bạn tự quyết định ảnh, tham số và tự kết luận. Xem HUONG_DAN.md để biết lệnh terminal và đường dẫn.

Phép quay: +30 độ, giữ kích thước khung. Phép phóng đại: affine x'=3x, y'=3y, kích thước ảnh đích 3H,3W. Cùng một ảnh gốc được dùng cho cả hai phép kiểm chứng.

## Cài đặt — KHÔNG dùng AI

Cell trống sau dành cho các định nghĩa hàm do nhóm tự viết: harris_classic, harris_laplace, repeatability và các hàm phụ thuộc. Không có nội dung thuật toán do AI cung cấp ở đây. Giao diện hàm cần đúng phần hướng dẫn.

## Chuẩn bị thư mục và cấu hình thí nghiệm

**Khai báo sử dụng AI — prompt nguyên văn:**

```text
[https://github.com/HoRa-W/ComputerVision-GroupF](https://github.com/HoRa-W/ComputerVision-GroupF)
hãy hướng dẫn tôi kiểm chứng phần cài đặt A ( 2 bất biến phép quay, quay ảnh 30 độ và 3 tỉ lệ) một cách kỹ càng (terminal trong vs studio code chạy những gì, file ảnh nên để ảnh ra sao như thế nào
```

Phạm vi hỗ trợ: môi trường, đọc/biến đổi ảnh, gọi hàm nhóm, lưu bảng và vẽ đối chiếu. Khung không cung cấp detector hoặc repeatability.

In [1]:
from pathlib import Path
import os
import sys

notebook_cwd = Path.cwd().resolve()
if notebook_cwd.name == "BTVN2":
    EXPERIMENT_DIR = notebook_cwd
elif (notebook_cwd / "BTVN2").is_dir():
    EXPERIMENT_DIR = notebook_cwd / "BTVN2"
else:
    raise RuntimeError("Hãy mở project ComputerVision-GroupF và đặt notebook trong BTVN2.")
os.chdir(EXPERIMENT_DIR)
print("Python:", sys.executable)
print("Thư mục:", EXPERIMENT_DIR)
print("Ảnh tồn tại:", (EXPERIMENT_DIR / "anh/kiem_chung_goc.jpg").is_file())

# Chọn theo hợp đồng đầu vào của harris_classic do nhóm tự viết.
CLASSIC_INPUT = "uint8"  # đổi thành "float01" nếu hàm nhóm nhận float [0,1]
COMMON = ["--classic-module", "__main__",
          "--laplace-module", "__main__",
          "--metric-module", "__main__",
          "--classic-input", CLASSIC_INPUT,
          "--image", "anh/kiem_chung_goc.jpg"]


RuntimeError: Hãy mở project ComputerVision-GroupF và đặt notebook trong BTVN2.

## Khung điều phối và hiển thị thí nghiệm

**Khai báo sử dụng AI — prompt nguyên văn:**

```text
[https://github.com/HoRa-W/ComputerVision-GroupF](https://github.com/HoRa-W/ComputerVision-GroupF)
hãy hướng dẫn tôi kiểm chứng phần cài đặt A ( 2 bất biến phép quay, quay ảnh 30 độ và 3 tỉ lệ) một cách kỹ càng (terminal trong vs studio code chạy những gì, file ảnh nên để ảnh ra sao như thế nào 
```

Phạm vi hỗ trợ: môi trường, đọc/biến đổi ảnh, gọi hàm nhóm, lưu bảng và vẽ đối chiếu. Khung không cung cấp detector hoặc repeatability.

In [ ]:
"""Khung thí nghiệm A.2/A.3; không cài đặt detector hoặc repeatability.

Đặt cạnh harris.py. Các hàm thuật toán và phép đo phải do nhóm tự viết.
Ảnh, điểm phát hiện và báo cáo được lưu để kiểm tra lại.
"""
from pathlib import Path
import argparse
import csv
import importlib
import inspect
import json
import sys

import numpy as np


ROOT = EXPERIMENT_DIR
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


def make_parser():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--test", choices=["check", "rotation", "scale", "all"], default="check")
    p.add_argument("--image", default="anh/kiem_chung_goc.jpg")
    p.add_argument("--out", default="ket_qua_A")
    p.add_argument("--classic-module", default="harris")
    p.add_argument("--laplace-module", default="harris")
    p.add_argument("--metric-module", default="harris")
    p.add_argument("--classic-input", choices=["uint8", "float01"], default="uint8")
    p.add_argument("--sigma-i", type=float, default=2.0)
    p.add_argument("--sigma-d", type=float, default=1.4)
    p.add_argument("--k", type=float, default=0.04)
    p.add_argument("--thresh-rel", type=float, default=0.01)
    p.add_argument("--sigma0", type=float, default=1.5)
    p.add_argument("--factor", type=float, default=1.2)
    p.add_argument("--levels", type=int, default=18)
    p.add_argument("--log-thresh", type=float, default=0.01)
    p.add_argument("--eps", type=float, default=3.0)
    return p


def task_path(value):
    path = Path(value)
    return path if path.is_absolute() else ROOT / path


def get_group_functions(args):
    specs = [(args.classic_module, "harris_classic"),
             (args.metric_module, "repeatability")]
    if args.test != "rotation":
        specs.append((args.laplace_module, "harris_laplace"))
    functions = {}
    missing = []
    for module_name, name in specs:
        module = importlib.import_module(module_name)
        fn = getattr(module, name, None)
        if callable(fn):
            functions[name] = fn
            print(f"CO: {module_name}.{name}{inspect.signature(fn)}")
        else:
            missing.append(f"{module_name}.{name}")
            print(f"THIEU: {module_name}.{name}")
        print(f"  File: {getattr(module, '__file__', '(khong ro)')}")
    if missing:
        print("CHUA SAN SANG. Nhom can tu hoan thien cac ham con thieu.")
        print("Khung nay khong tao, thay the hoac sua cac ham do.")
        return None
    return functions


def detected_points(fn, method, gray_u8, args):
    """Gọi hàm nhóm và chuyển quy ước đầu ra cho phần hiển thị/đo đạc."""
    if method == "harris_classic":
        image = gray_u8
        if args.classic_input == "float01":
            image = gray_u8.astype(np.float64) / 255.0
        raw = fn(image, sigma_i=args.sigma_i, sigma_d=args.sigma_d,
                 k=args.k, thresh_rel=args.thresh_rel)
        columns = ["row", "col", "response"]
        order = [0, 1]
    else:
        parameters = inspect.signature(fn).parameters
        if "sigmas" in parameters:
            sigmas = args.sigma0 * args.factor ** np.arange(args.levels)
            raw = fn(gray_u8, sigmas=sigmas, k=args.k,
                     thresh_rel=args.thresh_rel, log_thresh=args.log_thresh)
            columns = ["row", "col", "sigma", "response"]
            order = [0, 1]
        elif "so_thang_do" in parameters:
            # API hiện tại của BTVN2/harris.py: uint8 -> (x, y, sigma).
            raw = fn(gray_u8, so_thang_do=args.levels, sigma_0=args.sigma0,
                     factor=args.factor, nguong_log=args.log_thresh)
            columns = ["x", "y", "sigma"]
            order = [1, 0]
            print("  API harris_laplace hien tai: k=0.04, nguong Harris=0.01;")
            print("  hai tham so nay nam trong ham cua nhom, khong duoc doi boi khung.")
        else:
            raise ValueError("API harris_laplace khac hai API da mo ta; nhom can kiem tra giao dien.")
    array = np.asarray(raw, dtype=float)
    if array.size == 0:
        array = np.empty((0, len(columns)))
    if array.ndim != 2 or array.shape[1] != len(columns):
        raise ValueError(f"Dau ra {method} can cac cot {columns}; nhan shape={array.shape}.")
    if not np.isfinite(array).all():
        raise ValueError(f"Dau ra {method} chua NaN/Inf.")
    return array[:, order], array, columns


def project_rc(points, matrix):
    """Áp dụng ma trận tạo ảnh cho tọa độ (row, col), không ghép điểm."""
    xy1 = np.column_stack((points[:, 1], points[:, 0], np.ones(len(points))))
    xy = xy1 @ matrix.T
    return xy[:, [1, 0]]


def inside(points, shape):
    h, w = shape[:2]
    return ((points[:, 0] >= 0) & (points[:, 0] < h)
            & (points[:, 1] >= 0) & (points[:, 1] < w))


def save_csv(path, headers, rows):
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(rows)


def overlay(path, source, target, pa, pb, projected, valid, title):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    axes[0].imshow(source, cmap="gray", vmin=0, vmax=255)
    axes[0].scatter(pa[:, 1], pa[:, 0], s=22, facecolors="none", edgecolors="lime")
    axes[0].set_title(f"Anh goc: {len(pa)} diem")
    axes[1].imshow(target, cmap="gray", vmin=0, vmax=255)
    pv = projected[valid]
    axes[1].scatter(pv[:, 1], pv[:, 0], s=40, facecolors="none",
                    edgecolors="cyan", label="Vi tri du doan T(P_A)")
    axes[1].scatter(pb[:, 1], pb[:, 0], s=16, marker="x", color="orange",
                    label="Phat hien lai P_B")
    axes[1].set_title(f"Anh dich: {len(pb)} diem phat hien")
    axes[1].legend(loc="best", fontsize=8)
    for ax in axes:
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(path, dpi=170)
    plt.close(fig)


def experiment(kind, args, functions, source, out):
    import cv2
    h, w = source.shape
    if kind == "rotation":
        matrix = cv2.getRotationMatrix2D(((w - 1) / 2, (h - 1) / 2), 30.0, 1.0)
        size = (w, h)
        methods = ["harris_classic"]
        stem = "quay30"
    else:
        # Affine phóng đại quanh gốc: x'=3x, y'=3y; không dùng quy ước resize.
        matrix = np.array([[3.0, 0.0, 0.0], [0.0, 3.0, 0.0]])
        size = (3 * w, 3 * h)
        methods = ["harris_classic", "harris_laplace"]
        stem = "tile3"
    target = cv2.warpAffine(source, matrix, size, flags=cv2.INTER_LINEAR,
                            borderMode=cv2.BORDER_CONSTANT, borderValue=0)
    if not cv2.imwrite(str(out / f"anh_{stem}.png"), target):
        raise OSError("Khong luu duoc anh bien doi.")
    config = dict(vars(args))
    config.update(source_shape=list(source.shape), target_shape=list(target.shape),
                  transform_xy=matrix.tolist(), interpolation="INTER_LINEAR",
                  border="constant=0", eps_units="pixels in target image",
                  validity="0 <= row < H_target; 0 <= col < W_target",
                  definition="Use the group's repeatability; no matching implemented here.",
                  numpy_version=np.__version__, opencv_version=cv2.__version__)
    config["modules"] = {}
    for name, fn in functions.items():
        config["modules"][name] = {"file": inspect.getsourcefile(fn),
                                    "signature": str(inspect.signature(fn))}
    (out / f"cau_hinh_{stem}.json").write_text(
        json.dumps(config, ensure_ascii=False, indent=2), encoding="utf-8")
    rows = []
    for method in methods:
        print(f"DANG CHAY: {stem} / {method}")
        pa, raw_a, cols_a = detected_points(functions[method], method, source, args)
        pb, raw_b, cols_b = detected_points(functions[method], method, target, args)
        projected = project_rc(pa, matrix)
        valid = inside(projected, target.shape)
        n_valid = int(valid.sum())
        if n_valid == 0:
            rep = None
            print("KHONG DANH GIA DUOC: khong co diem goc hop le.")
        else:
            def transform(rc):
                return tuple(project_rc(np.asarray([rc], dtype=float), matrix)[0])
            # Chỉ gọi phép đo do nhóm tự cài đặt; không cài đặt phép ghép điểm.
            rep = float(functions["repeatability"](
                pa, pb, transform, target.shape, eps=args.eps))
            if not np.isfinite(rep) or not 0 <= rep <= 1:
                raise ValueError("repeatability cua nhom phai tra ve mot so trong [0, 1].")
        prefix = f"{stem}_{method}"
        save_csv(out / f"{prefix}_goc.csv", cols_a, raw_a)
        save_csv(out / f"{prefix}_dich.csv", cols_b, raw_b)
        save_csv(out / f"{prefix}_du_doan.csv", ["row_dich", "col_dich", "hop_le"],
                 np.column_stack((projected, valid.astype(int))))
        text_rep = "N/A" if rep is None else f"{rep:.6f} ({100 * rep:.2f}%)"
        print(f"{method}: N_A={len(pa)}; N_B={len(pb)}; N_valid={n_valid}; repeatability={text_rep}")
        overlay(out / f"{prefix}_doi_chieu.png", source, target, pa, pb,
                projected, valid, f"{stem} / {method} / repeatability = {text_rep}")
        rows.append([method, len(pa), len(pb), n_valid, "" if rep is None else rep, args.eps])
    save_csv(out / f"bao_cao_{stem}.csv",
             ["method", "N_A", "N_B", "N_valid", "repeatability", "eps_target_px"], rows)


def run(args):
    print(f"Python: {sys.executable}")
    print(f"Thu muc khung: {ROOT}")
    if args.levels < 3 or args.sigma0 <= 0 or args.factor <= 1 or args.eps <= 0:
        raise ValueError("Can levels>=3, sigma0>0, factor>1, eps>0.")
    functions = get_group_functions(args)
    if functions is None:
        return 2
    if args.test == "check":
        print("DU TEN HAM. Can tu kiem chung tinh dung cua cac ham truoc khi ket luan.")
        return 0
    import cv2
    image_path = task_path(args.image)
    out = task_path(args.out)
    if not image_path.is_file():
        raise FileNotFoundError(f"Khong co anh: {image_path}")
    source = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if source is None:
        raise ValueError(f"OpenCV khong doc duoc anh: {image_path}")
    out.mkdir(parents=True, exist_ok=True)
    if not cv2.imwrite(str(out / "anh_goc_xam.png"), source):
        raise OSError("Khong luu duoc anh xam.")
    print(f"Anh: {image_path}; kich thuoc H,W={source.shape}; dtype={source.dtype}")
    if args.test in ("rotation", "all"):
        experiment("rotation", args, functions, source, out)
    if args.test in ("scale", "all"):
        experiment("scale", args, functions, source, out)
    print(f"DA LUU: {out}")
    return 0


def main():
    args = make_parser().parse_args()
    try:
        return run(args)
    except Exception as exc:
        print(f"DUNG: {type(exc).__name__}: {exc}", file=sys.stderr)
        print("Kiem tra giao dien/du lieu; khong tu coi lan chay nay la da qua kiem chung.", file=sys.stderr)
        return 1



## Kiểm tra các hàm nhóm đã có

**Khai báo sử dụng AI — prompt nguyên văn:**

```text
[https://github.com/HoRa-W/ComputerVision-GroupF](https://github.com/HoRa-W/ComputerVision-GroupF)
hãy hướng dẫn tôi kiểm chứng phần cài đặt A ( 2 bất biến phép quay, quay ảnh 30 độ và 3 tỉ lệ) một cách kỹ càng (terminal trong vs studio code chạy những gì, file ảnh nên để ảnh ra sao như thế nào 
```

Phạm vi hỗ trợ: môi trường, đọc/biến đổi ảnh, gọi hàm nhóm, lưu bảng và vẽ đối chiếu. Khung không cung cấp detector hoặc repeatability.

In [ ]:
status = run(make_parser().parse_args(["--test", "check"] + COMMON))
if status != 0:
    raise RuntimeError("Nhóm chưa có đủ hàm trong notebook; xem các dòng THIEU ở trên.")


## A.2 — Quay 30 độ

**Khai báo sử dụng AI — prompt nguyên văn:**

```text
[https://github.com/HoRa-W/ComputerVision-GroupF](https://github.com/HoRa-W/ComputerVision-GroupF)
hãy hướng dẫn tôi kiểm chứng phần cài đặt A ( 2 bất biến phép quay, quay ảnh 30 độ và 3 tỉ lệ) một cách kỹ càng (terminal trong vs studio code chạy những gì, file ảnh nên để ảnh ra sao như thế nào 
```

Phạm vi hỗ trợ: môi trường, đọc/biến đổi ảnh, gọi hàm nhóm, lưu bảng và vẽ đối chiếu. Khung không cung cấp detector hoặc repeatability.

In [ ]:
status = run(make_parser().parse_args(["--test", "rotation"] + COMMON))
if status != 0:
    raise RuntimeError("Kiểm chứng quay chưa chạy xong.")


## A.3 — Phóng to 3 lần

**Khai báo sử dụng AI — prompt nguyên văn:**

```text
[https://github.com/HoRa-W/ComputerVision-GroupF](https://github.com/HoRa-W/ComputerVision-GroupF)
hãy hướng dẫn tôi kiểm chứng phần cài đặt A ( 2 bất biến phép quay, quay ảnh 30 độ và 3 tỉ lệ) một cách kỹ càng (terminal trong vs studio code chạy những gì, file ảnh nên để ảnh ra sao như thế nào 
```

Phạm vi hỗ trợ: môi trường, đọc/biến đổi ảnh, gọi hàm nhóm, lưu bảng và vẽ đối chiếu. Khung không cung cấp detector hoặc repeatability.

In [ ]:
status = run(make_parser().parse_args([
    "--test", "scale", "--sigma0", "1.5", "--factor", "1.2", "--levels", "18"
] + COMMON))
if status != 0:
    raise RuntimeError("Kiểm chứng tỉ lệ chưa chạy xong.")


## Xem lại bảng và hình kết quả

**Khai báo sử dụng AI — prompt nguyên văn:**

```text
[https://github.com/HoRa-W/ComputerVision-GroupF](https://github.com/HoRa-W/ComputerVision-GroupF)
hãy hướng dẫn tôi kiểm chứng phần cài đặt A ( 2 bất biến phép quay, quay ảnh 30 độ và 3 tỉ lệ) một cách kỹ càng (terminal trong vs studio code chạy những gì, file ảnh nên để ảnh ra sao như thế nào 
```

Phạm vi hỗ trợ: môi trường, đọc/biến đổi ảnh, gọi hàm nhóm, lưu bảng và vẽ đối chiếu. Khung không cung cấp detector hoặc repeatability.

In [ ]:
from IPython.display import display, Image

for filename in ["bao_cao_quay30.csv", "bao_cao_tile3.csv"]:
    report = ROOT / "ket_qua_A" / filename
    if not report.is_file():
        raise FileNotFoundError(f"Chưa có kết quả: {report}")
    print(filename)
    print(report.read_text(encoding="utf-8-sig"))

for filename in ["quay30_harris_classic_doi_chieu.png",
                 "tile3_harris_classic_doi_chieu.png",
                 "tile3_harris_laplace_doi_chieu.png"]:
    display(Image(filename=str(ROOT / "ket_qua_A" / filename)))


## Nhận xét — nhóm tự viết từ kết quả thực tế

- Ảnh được chọn và lý do chọn: …
- Tham số thực tế, quy ước đầu vào, dải sigma: …
- Kết quả quay: số điểm, repeatability, quan sát vị trí: …
- Kết quả tỉ lệ: hai repeatability và quan sát vị trí: …
- Liên hệ kết quả với cơ chế thuật toán; giới hạn cấu hình/dữ liệu: …

Sau khi tích hợp vào notebook duy nhất của bài nộp, chọn Restart & Run All và lưu output. Nộp thư mục anh/ cùng notebook. Nếu sử dụng thêm AI, bổ sung nguyên văn prompt tương ứng ngay trước cell có dùng hỗ trợ đó.